# 01 · Dataset overview

This notebook establishes the data facts and classification framing shared by the Logistic Regression and KNN tracks. The target is `diagnosis`; `id` is a record identifier and is excluded from candidate predictors. No model is fitted here.

**Proposed decision metric:** malignant-class recall, with benign/malignant labels shown explicitly in all reports. Missing a malignant case is the higher-cost error in this screening-oriented exercise. Precision, F1, and ROC-AUC are secondary diagnostics. This is a research exercise, not a clinical diagnostic tool.

**Validation plan for later modeling:** stratified partitions because records are non-temporal and the classes are imbalanced; all learned preprocessing must be fitted only on training partitions. Keep the test partition untouched until final evaluation.

**Known risks to audit:** modest sample size, correlated measurements, potential ID leakage, class imbalance, duplicates, missing/invalid values, and split sensitivity. A single split is not enough to establish robustness; use stratified cross-validation on training data when model-building begins.

In [ ]:
from pathlib import Path
import random
import platform
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_PATH = Path('../data/raw/Raw_Breast_Cancer_Data.csv')
TARGET = 'diagnosis'
ID_COLUMN = 'id'
print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__} | NumPy: {np.__version__}')
print(f'Seed: {SEED}')

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}')
print(f'DataFrame memory: {df.memory_usage(deep=True).sum() / 1024**2:.3f} MiB')
display(df.head())
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique(dropna=False)}))

In [ ]:
print('Dtype counts:')
display(df.dtypes.astype(str).value_counts().rename('columns').to_frame())
print('Numeric summary:')
display(df.describe().T)
print('Target distribution:')
target_counts = df[TARGET].value_counts(dropna=False).rename_axis(TARGET).to_frame('count')
target_counts['fraction'] = target_counts['count'] / len(df)
display(target_counts)
print('ID unique:', df[ID_COLUMN].nunique(), 'of', len(df))

In [ ]:
quality = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'missing_cells': int(df.isna().sum().sum()),
    'all_missing_columns': int(df.isna().all().sum()),
    'duplicate_rows': int(df.duplicated().sum()),
    'duplicate_ids': int(df[ID_COLUMN].duplicated().sum()),
    'non_numeric_predictor_columns': int(df.drop(columns=[TARGET, ID_COLUMN]).select_dtypes(exclude='number').shape[1]),
})
display(quality.to_frame('value'))
print('Target labels:', sorted(df[TARGET].dropna().unique().tolist()))
print('Non-positive numeric feature values (inspect context; zero can be valid):', int((df.drop(columns=[TARGET, ID_COLUMN]).select_dtypes(include='number') <= 0).sum().sum()))
empty_columns = df.columns[df.isna().all()].tolist()
print('All-missing columns to exclude from later feature matrices:', empty_columns)

## Initial feasibility and risk assessment

The source CSV has 569 rows and 33 columns based on the initial inspection; this notebook recomputes those facts from the loaded file. That is enough to demonstrate regularized classical classifiers, but it is still a modest dataset: expect split-to-split uncertainty and avoid claims of clinical readiness. The target is binary and categorical. Use scaling inside a train-fitted pipeline for both Logistic Regression and distance-based KNN.

`id` must not be used as a feature. Correlation among the radius, perimeter, and area measurements (and between mean, standard-error, and worst measurements) is plausible and will be checked during visualization/modeling. The CSV includes an entirely empty `Unnamed: 32` column (569 missing cells); it carries no information and should be excluded from feature matrices while leaving the raw CSV unchanged. Duplicate rows and unusual values are quantified above before mitigation is considered. Zero is allowed for some shape measurements, so non-positive counts require feature-level interpretation rather than automatic row removal. There is no stated time column, so random stratification is the current validation proposal; this assumption should be revisited if row ordering or provenance reveals grouping or temporal structure.

### Self-critique

The strongest objection is that malignant recall as the sole primary metric ignores false positives and the dataset's sampling context; the dataset may not represent the prevalence or clinical costs of deployment. It is selected because this project is explicitly framed around breast-cancer classification and missed malignant cases warrant attention, but this does not make a model clinically useful. Later reports must show precision, confusion matrices, uncertainty/stability, and clearly state this limitation.